In [1]:
import sys
!"{sys.executable}" -m pip uninstall opencv-python -y
!"{sys.executable}" -m pip uninstall opencv-contrib-python -y
!"{sys.executable}" -m pip install opencv-contrib-python --user

Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl (46.5 MB)


In [2]:
import cv2
print(hasattr(cv2, "face"))

True


In [4]:
import cv2
import numpy as np

# load data
try:
    data = np.load("faces.npy", allow_pickle=True).item()
except:
    print("❌ No data found! Run collection first.")
    exit()

faces = []
labels = []
names = {}
label_id = 0

# prepare training data
for name, images in data.items():
    names[label_id] = name
    for img in images:
        faces.append(img)
        labels.append(label_id)
    label_id += 1

faces = np.array(faces)
labels = np.array(labels)

# LBPH model (requires opencv-contrib-python)
model = cv2.face.LBPHFaceRecognizer_create()
model.train(faces, labels)

cap = cv2.VideoCapture(0)

classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if classifier.empty():
    print("Error loading cascade file")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        continue

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    detected = classifier.detectMultiScale(gray, 1.2, 6)

    for (x, y, w, h) in detected:
        face = gray[y:y+h, x:x+w]
        face = cv2.resize(face, (100, 100))

        label, confidence = model.predict(face)

        name = names[label]

        # confidence threshold (IMPORTANT)
        if confidence < 80:
            text = f"{name} ({int(confidence)})"
        else:
            text = "Unknown"

        cv2.rectangle(frame, (x,y), (x+w,y+h), (255,0,0), 2)
        cv2.putText(frame, text, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1,
                    (0,255,0), 2)

    cv2.imshow("Face Recognition", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
        break

cap.release()
cv2.destroyAllWindows()